# Vision basics

**`Retina`** maps RGB frames to ommatidium channels; **`SimpleVisualTaxisController`** combines that with **`HybridTurningController`** for object-centered turning.

The shipped compound-eye assets expect 512×450 frames. Here we use a **toy 2×2 ommatidia map** (same idea as unit tests) and a matching low-res render so the notebook runs with default MuJoCo framebuffer settings. For full resolution, instantiate `Retina()` with defaults, set `camera_res=(retina.nrows, retina.ncols)`, and increase `mj_model.vis.global_.offheight` / `offwidth` before `set_renderer`.

**`MovingFlyWorld`** still moves a dark target in the scene for realism; the toy drive below scales left vs right eye readings so turning is visible.


In [ ]:
import numpy as np
from flygym import Simulation
from flygym.anatomy import ContactBodiesPreset
from flygym.utils.math import Rotation3D
from flygym_demo.examples import make_walking_fly, run_closed_loop
from flygym_demo.examples.locomotion import HybridTurningController
from flygym_demo.examples.vision import Retina, SimpleVisualTaxisController
from flygym_demo.examples.worlds import MovingFlyWorld

fly = make_walking_fly(add_camera=True)
trackcam = fly.cameraname_to_mjcfcamera["trackcam"]
world = MovingFlyWorld(init_pos=(4.0, 0.0, 1.8), speed=3.0, lateral_magnitude=0.15)
world.add_fly(
    fly,
    spawn_position=[0, 0, 1.5],
    spawn_rotation=Rotation3D("quat", [1, 0, 0, 0]),
    bodysegs_with_ground_contact=ContactBodiesPreset.TIBIA_TARSUS_ONLY,
    add_ground_contact_sensors=False,
)
sim = Simulation(world)
retina = Retina(
    ommatidia_id_map=np.array([[0, 1], [2, 2]], dtype=np.int16),
    pale_type_mask=np.array([0, 1]),
    nrows=2,
    ncols=2,
)
sim.set_renderer(trackcam, camera_res=(retina.nrows, retina.ncols))
cam_key = trackcam.full_identifier
base = HybridTurningController(sim.timestep)
controller = SimpleVisualTaxisController(base, retina, gain=0.9)


def drive_fn(step_idx, t, sim):
    sim.render_as_needed()
    if sim.renderer is None or sim.renderer.frames is None:
        return None
    frames = sim.renderer.frames.get(cam_key, [])
    if not frames:
        return None
    rgb = np.asarray(frames[-1])
    undist = retina.correct_fisheye(rgb)
    hex_eyes = retina.raw_image_to_hex_pxls(undist)
    w_l = 1.0 + 0.35 * np.sin(2.0 * np.pi * 1.5 * t)
    w_r = 1.0 - 0.35 * np.sin(2.0 * np.pi * 1.5 * t)
    return np.stack([hex_eyes * w_l, hex_eyes * w_r], axis=0)


if hasattr(world, "reset"):
    world.reset(sim)
records = run_closed_loop(sim, controller, 0.01, fly_name=fly.name, drive_fn=drive_fn, render=True)
print("vision_turn_bias", records[-1]["metadata"].get("vision_turn_bias"))
